In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import *
from delta.tables import DeltaTable

### **Parameters**

In [0]:
dimension_config = {
    "customers": {
        "source_object": "silver_customers",
        "source_schema":"silver_ecommerce",
        "target_schema":"gold_ecommerce",
        "target_object": "DimCustomers",
        "key_cols": ["customer_id"],
        "cdc_col":"modified_date",
        "surrogate_key": "DIMCustomersKey"
    },

    "products": {
        "source_object": "silver_products",
        "source_schema":"silver_ecommerce",
        "target_schema":"gold_ecommerce",
        "target_object": "DimProducts",
        "key_cols": ["product_id"],
        "cdc_col":"modified_date",
        "surrogate_key": "DIMProductsKey"
    }
}

In [0]:
dbutils.widgets.text("src","")
src_value=dbutils.widgets.get("src")
print(src_value)
config=dimension_config[src_value]
print(config)

In [0]:
key_cols_list = config["key_cols"]

cdc_col = config["cdc_col"]

backdated_col = ''

source_schema = config["source_schema"]
source_object = config["source_object"]

target_schema = config["target_schema"]
target_object = config["target_object"]

surrogate_key = config["surrogate_key"]

In [0]:
if len(backdated_col)==0:
    if spark.catalog.tableExists(f"{target_schema}.{target_object}"):
        last_load=spark.sql(f"select max({cdc_col}) from {target_schema}.{target_object}").collect()[0][0]
    else:
        last_load='1900-01-01 00:00:00'
else:
    last_load=backdated_col
print(last_load)
    


### **Incremental Ingestion**

In [0]:
df_source=spark.sql(f"select * from {source_schema}.{source_object} where {cdc_col}>'{last_load}'")

In [0]:
if spark.catalog.tableExists(f"{target_schema}.{target_object}"):
    key_cols_initial=','.join(key_cols_list)
    df_target=spark.sql(f"select {key_cols_initial},{surrogate_key},trg_created_date,trg_updated_date from {target_schema}.{target_object}")
else:
    key_cols_incremental=[f" '' as {i}"  for i in key_cols_list]
    key_cols_incremental=','.join(key_cols_incremental)
    df_target=spark.sql(f"select {key_cols_incremental},cast('0' as int) as {surrogate_key},cast('1900-01-01 00:00:00' as timestamp) as trg_created_date,cast('1900-01-01 00:00:00' as timestamp) as trg_updated_date where 1 = 0")



In [0]:
df_target.show()

In [0]:
join_condition='AND'.join(f"src.{i}=trg.{i}"for i in key_cols_list)

In [0]:
df_source.createOrReplaceTempView("src")
df_target.createOrReplaceTempView("trg")
trg_key_cols = ','.join([f"trg.{i}" for i in key_cols_list])
df_join=spark.sql(f"""
                  select src.*,
                  trg.{surrogate_key} as {surrogate_key},
                  trg.trg_created_date,
                  trg.trg_updated_date
                  from src
                  left join trg
                  on 
                  {join_condition}
                  """)

In [0]:
df_old=df_join.filter(f"trg.{surrogate_key} is not null")
df_new=df_join.filter(f"trg.{surrogate_key} is null")
df_old.show()
df_new.show()

In [0]:
df_old_enriched=df_old.withColumn("trg_updated_date",current_timestamp())

In [0]:
df_old_enriched.show()

### **Finding Surrogate Key**

In [0]:
if spark.catalog.tableExists(f"{target_schema}.{target_object}"):
    max_surrogate_key=spark.sql(f"select max({surrogate_key}) from {target_schema}.{target_object}").collect()[0][0]
    df_new_enriched=df_new.withColumn("trg_created_date",current_timestamp())\
        .withColumn("trg_updated_date",current_timestamp())\
        .withColumn(f"{surrogate_key}",max_surrogate_key+lit(1)+monotonically_increasing_id())
else:
    max_surrogate_key=0
    df_new_enriched=df_new.withColumn("trg_created_date",current_timestamp())\
        .withColumn("trg_updated_date",current_timestamp())\
        .withColumn(f"{surrogate_key}",max_surrogate_key+lit(1)+monotonically_increasing_id())

df_new_enriched.show()  

In [0]:
df_union=df_new_enriched.unionByName(df_old_enriched)

In [0]:
display(df_union)

In [0]:
if spark.catalog.tableExists(f"{target_schema}.{target_object}"):
    dlt_trg=DeltaTable.forName(spark, f"{target_schema}.{target_object}")
    dlt_trg.alias("trg").merge(df_union.alias("src"),f"trg.{surrogate_key}==src.{surrogate_key}").whenMatchedUpdateAll(condition=f"src.{cdc_col}>trg.{cdc_col}").whenNotMatchedInsertAll().execute()
else:
    df_union.write.format("delta").mode("overwrite").saveAsTable(f"{target_schema}.{target_object}")